In [ ]:
%%capture
!pip install unsloth datasets scikit-learn pandas accelerate

In [ ]:
import json
import re
import numpy as np
import pandas as pd
from typing import Iterator, Dict

from datasets import Dataset
from sklearn.model_selection import KFold
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_NAME      = "unsloth/gemma-3-1b-it"
MAX_SEQ_LENGTH  = 1024
LOAD_IN_4BIT    = True

# ── LoRA ─────────────────────────────────────────────────────────────────────
LORA_R          = 16
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.0
TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS          = 3
BATCH_SIZE      = 4
GRAD_ACCUM      = 4
LR              = 2e-4
WARMUP_RATIO    = 0.05
OUTPUT_DIR      = "./outputs"

# ── Splits ───────────────────────────────────────────────────────────────────
SEED:      int   = 42
N_FOLDS:   int   = 3

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "Your task is to extract address components from the user's message. "
    "Ignore any missing information and exclude those fields from your output. "
    "Return the data in JSON format using this structure:\n"
    "Output format:\n"
    "{\n"
    '  "house_number": "<house number>",\n'
    '  "street": "<street name>",\n'
    '  "city": "<city name>",\n'
    '  "postal_code": "<postal or zip code>",\n'
    '  "state": "<state or province name>",\n'
    '  "country": "<country name>"\n'
    "}"
)

### Dataset

In [ ]:
from data_preparation.preprocess_data import make_splits, preprocess_data

df = preprocess_data()

### Chat-template formatting

In [ ]:
def df_to_hf_dataset(df: pd.DataFrame, tokenizer) -> Dataset:
    """Convert DataFrame rows → formatted chat strings via the tokenizer chat template."""

    def _format(row):
        # Drop empty fields from target
        target = {k: v for k, v in row["target"].items() if v}
        messages = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": row["input"]},
            {"role": "assistant", "content": json.dumps(target, ensure_ascii=False)},
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

    texts = df.apply(_format, axis=1).tolist()
    return Dataset.from_dict({"text": texts})

### Training loop over folds

In [ ]:
fold_results = []

for fold_idx, split in enumerate(make_splits(df)):
    print(f"\n{'='*60}")
    print(f" FOLD {fold_idx + 1} / {N_FOLDS}")
    print(f"  train={len(split['train'])}  val={len(split['val'])}  test={len(split['test'])}")
    print(f"{'='*60}")

    # ── Load model & tokenizer (fresh each fold) ──────────────────────────────
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        load_in_4bit   = LOAD_IN_4BIT,
        dtype          = None,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r              = LORA_R,
        target_modules = TARGET_MODULES,
        lora_alpha     = LORA_ALPHA,
        lora_dropout   = LORA_DROPOUT,
        bias           = "none",
        use_gradient_checkpointing = "unsloth",
        random_state   = SEED,
    )

    train_ds = df_to_hf_dataset(split["train"], tokenizer)
    val_ds   = df_to_hf_dataset(split["val"],   tokenizer)

    fold_output_dir = f"{OUTPUT_DIR}/fold_{fold_idx + 1}"

    trainer = SFTTrainer(
        model          = model,
        tokenizer      = tokenizer,
        train_dataset  = train_ds,
        eval_dataset   = val_ds,
        args           = SFTConfig(
            output_dir                  = fold_output_dir,
            num_train_epochs            = EPOCHS,
            per_device_train_batch_size = BATCH_SIZE,
            per_device_eval_batch_size  = BATCH_SIZE,
            gradient_accumulation_steps = GRAD_ACCUM,
            learning_rate               = LR,
            warmup_ratio                = WARMUP_RATIO,
            lr_scheduler_type           = "cosine",
            fp16                        = not FastLanguageModel.is_bfloat16_supported(),
            bf16                        = FastLanguageModel.is_bfloat16_supported(),
            logging_steps               = 20,
            eval_strategy               = "epoch",
            save_strategy               = "epoch",
            load_best_model_at_end      = True,
            metric_for_best_model       = "eval_loss",
            greater_is_better           = False,
            dataset_text_field          = "text",
            max_seq_length              = MAX_SEQ_LENGTH,
            packing                     = False,
            seed                        = SEED,
        ),
    )

    trainer.train()

    # ── Quick eval on test split ───────────────────────────────────────────────
    test_ds    = df_to_hf_dataset(split["test"], tokenizer)
    test_metrics = trainer.evaluate(test_ds)
    print(f"Fold {fold_idx + 1} test metrics: {test_metrics}")
    fold_results.append(test_metrics)

    # ── Save LoRA adapter ─────────────────────────────────────────────────────
    model.save_pretrained(f"{fold_output_dir}/lora_adapter")
    tokenizer.save_pretrained(f"{fold_output_dir}/lora_adapter")
    print(f"LoRA adapter saved to {fold_output_dir}/lora_adapter")

    # Free VRAM before next fold
    del model, tokenizer, trainer
    import gc, torch
    gc.collect()
    torch.cuda.empty_cache()

### Eval

In [ ]:
print("\n=== Cross-fold test results ===")
for i, m in enumerate(fold_results):
    print(f"  Fold {i+1}: eval_loss={m.get('eval_loss', 'N/A'):.4f}")

avg_loss = np.mean([m.get("eval_loss", float("nan")) for m in fold_results])
print(f"\n  Mean test loss: {avg_loss:.4f}")